# Lab Assignment: English-to-German Machine Translation with a Transformer

**Objective:**
The goal of this assignment is to fine-tune a pre-trained **Transformer** model for machine translation. You will load a dataset, fine-tune the model, and evaluate it using the BLEU score.

You will also compare your new model's performance and architecture against a previous **LSTM-based** model, and visualize the **self-attention** and **cross-attention** mechanisms that make the Transformer so powerful.


## Part 1: Setup and Environment

First, install all the necessary libraries.

In [ ]:
# ---------------- Part 1: Setup and Environment ----------------
# All libraries needed for this notebook. transformers/datasets/evaluate/sacrebleu/
# sentencepiece/accelerate talk to the Hugging Face Hub and need internet access
# (available by default in Google Colab).
!pip install -q transformers datasets evaluate sacrebleu sentencepiece accelerate bertviz

import os
import random
import numpy as np
import pandas as pd
import torch

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    pipeline,
)
import evaluate

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)
if device == "cpu":
    print("WARNING: no GPU detected. In Colab go to Runtime > Change runtime type > T4 GPU, "
          "otherwise fine-tuning will be very slow.")

## Part 2: Data Loading and Pre-processing

### 1. Load the Dataset
We have to use previously provided dataset, which contains parallel sentences for English-to-German translation.

In [ ]:
# ---------------- Part 2.1: Load the Dataset ----------------
# Same source file used in the previous (LSTM) lab: deu.txt, tab-separated
# "English \t German \t attribution" lines from the Tatoeba/manythings.org project.
# We only need the first two columns (English, German).

def find_data_file():
    """Locate deu.txt, uploading or downloading it if it isn't present yet."""
    candidates = ['deu.txt', '/content/deu.txt', '/content/drive/MyDrive/deu.txt',
                  '/mnt/user-data/uploads/deu.txt']
    for p in candidates:
        if os.path.exists(p):
            return p
    try:
        from google.colab import files
        print("deu.txt not found - please upload it now ...")
        uploaded = files.upload()
        if 'deu.txt' in uploaded or os.path.exists('deu.txt'):
            return 'deu.txt'
    except ImportError:
        pass
    import urllib.request, zipfile
    print("Trying public download from manythings.org ...")
    urllib.request.urlretrieve('http://www.manythings.org/anki/deu-eng.zip', 'deu-eng.zip')
    with zipfile.ZipFile('deu-eng.zip') as zf:
        zf.extractall('.')
    return 'deu.txt'

data_path = find_data_file()
print("Using data file:", data_path)

# Use the same 10,000-sentence slice as the LSTM lab, so the two models are
# trained/compared on comparable data.
NUM_SAMPLES = 10000
pairs = []
with open(data_path, encoding='utf-8') as f:
    for line in f:
        if len(pairs) >= NUM_SAMPLES:
            break
        parts = line.rstrip('\n').split('\t')
        if len(parts) < 2:
            continue
        en, de = parts[0], parts[1]
        pairs.append({"en": en, "de": de})

print("Total pairs loaded:", len(pairs))
print("Example pair:", pairs[1000])

# Train / validation / test split: 80% / 10% / 10%
random.Random(42).shuffle(pairs)
n = len(pairs)
n_train = int(0.8 * n)
n_val = int(0.1 * n)
train_pairs = pairs[:n_train]
val_pairs = pairs[n_train:n_train + n_val]
test_pairs = pairs[n_train + n_val:]

raw_datasets = DatasetDict({
    "train": Dataset.from_list(train_pairs),
    "validation": Dataset.from_list(val_pairs),
    "test": Dataset.from_list(test_pairs),
})
print(raw_datasets)

### 2. Load Tokenizer and Model
We will use a pre-trained model from the Helsinki-NLP group, `opus-mt-en-de`, which is a lightweight and efficient Transformer specifically designed for English-to-German translation.

In [ ]:
# ---------------- Part 2.2: Load Tokenizer and Model ----------------
model_checkpoint = "Helsinki-NLP/opus-mt-en-de"

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint).to(device)

print("Model:", model_checkpoint)
print("Number of parameters: %.1fM" % (model.num_parameters() / 1e6))
print("Vocabulary size:", tokenizer.vocab_size)

### 3. Pre-processing Function
We must tokenize both the English (input) and German (target) sentences. For seq2seq models, the tokenized target sentences are assigned to the `labels` key.

In [ ]:
# ---------------- Part 2.3: Pre-processing Function ----------------
MAX_INPUT_LENGTH = 64
MAX_TARGET_LENGTH = 64

def preprocess_function(examples):
    inputs = examples["en"]
    targets = examples["de"]
    model_inputs = tokenizer(
        inputs, max_length=MAX_INPUT_LENGTH, truncation=True
    )
    # tokenize targets with the tokenizer's target mode so special tokens are handled correctly
    labels = tokenizer(
        text_target=targets, max_length=MAX_TARGET_LENGTH, truncation=True
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

### 4. Apply Pre-processing
Use the `.map()` function to apply this tokenization to the entire dataset.

In [ ]:
# ---------------- Part 2.4: Apply Pre-processing ----------------
tokenized_datasets = raw_datasets.map(
    preprocess_function,
    batched=True,
    remove_columns=raw_datasets["train"].column_names,
)
print(tokenized_datasets)
print("Example tokenized input ids:", tokenized_datasets["train"][0]["input_ids"])
print("Example tokenized labels   :", tokenized_datasets["train"][0]["labels"])

## Part 3: Fine-Tuning the Model

### 1. Define Evaluation Metric (BLEU)
Machine translation is evaluated using the **BLEU score**. You can load the `sacrebleu` metric and create a `compute_metrics` function to use during training.

In [ ]:
# ---------------- Part 3.1: Define Evaluation Metric (BLEU) ----------------
metric = evaluate.load("sacrebleu")

def postprocess_text(preds, labels):
    preds = [p.strip() for p in preds]
    # sacrebleu expects a list of references per prediction
    labels = [[l.strip()] for l in labels]
    return preds, labels

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    # replace -100 in the labels (used to mask padding for the loss) before decoding
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    result = {"bleu": result["score"]}

    pred_lens = [np.count_nonzero(np.array(p) != tokenizer.pad_token_id) for p in preds]
    result["gen_len"] = float(np.mean(pred_lens))
    result = {k: round(v, 4) for k, v in result.items()}
    return result

### 2. Configure Training
We will use the `Seq2SeqTrainer` and `Seq2SeqTrainingArguments` classes to manage the fine-tuning process.

In [ ]:
# ---------------- Part 3.2: Configure Training ----------------
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir="opus-mt-en-de-finetuned",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),   # mixed precision only makes sense on GPU
    load_best_model_at_end=True,
    metric_for_best_model="bleu",
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

### 3. Start Training


In [ ]:
# ---------------- Part 3.3: Start Training ----------------
train_result = trainer.train()

trainer.save_model("opus-mt-en-de-finetuned")
tokenizer.save_pretrained("opus-mt-en-de-finetuned")

print("Training complete. Final training metrics:")
print(train_result.metrics)

### 4. Evaluate the Model
After training, run the final evaluation on the test set.

In [ ]:
# ---------------- Part 3.4: Evaluate the Model ----------------
test_metrics = trainer.evaluate(eval_dataset=tokenized_datasets["test"], metric_key_prefix="test")
print("Test set results:")
for k, v in test_metrics.items():
    print(f"  {k}: {v}")

print("\nFinal Transformer test BLEU score: %.2f" % test_metrics["test_bleu"])

## Part 4: Inference (Testing the Translator)

Now use your fine-tuned model to translate new sentences using the `pipeline` utility.

In [ ]:
# ---------------- Part 4: Inference (Testing the Translator) ----------------
# Test with new sentences
text1 = "Hello, how are you doing today?"
text2 = "The transformer model is a powerful architecture for NLP."
text3 = "This assignment is for my machine learning lab."

translator = pipeline(
    "translation_en_to_de",
    model="opus-mt-en-de-finetuned",
    tokenizer="opus-mt-en-de-finetuned",
    device=0 if torch.cuda.is_available() else -1,
)

for t in (text1, text2, text3):
    out = translator(t)[0]["translation_text"]
    print(f"EN: {t}\nDE: {out}\n")

## Part 5: Comparative Analysis (Transformer vs. LSTM)

In this section, you will directly compare the output of your new Transformer model with the LSTM-based model from your previous lab.

### 5.1. Test Sentences
Here is a list of 10 English sentences varying in length and complexity.

In [ ]:
test_sentences = [
    # 1. Simple sentence
    "The cat sat on the mat.",

    # 2. Longer sentence
    "I am going to the library to read a book.",

    # 3. Question
    "What is your favorite color?",

    # 4. Long-range dependency
    "The man who lives down the street just bought a new car.",

    # 5. Figurative language (often difficult)
    "It's raining cats and dogs outside.",

    # 6. Technical term
    "The machine learning model was trained on a large dataset.",

    # 7. Command
    "Please close the door when you leave.",

    # 8. Classic test sentence
    "The quick brown fox jumps over the lazy dog.",

    # 9. Negation
    "He did not want to go to the party.",

    # 10. Future tense
    "We will travel to Germany next summer."
]


In [ ]:
# Real greedy-decoded output of the Lab 8 LSTM model on these same 10 sentences
# (produced by loading s2s_eng_ger.h5 and re-running its decode_sequence() function).
lstm_translations_lab8 = ['Es war schwimmen.', 'Ich bint mein Schürf.', 'Wer ist sie?', 'Ich bist Tom.', 'Ich bist Tom sieren Sie.', 'Ich gsterbeiche Tom.', 'Ich habe Tom gehenten.', 'Ich bint mein Schwimmen.', 'Ich bint mir da mit geschlafen.', 'Wir wirten eschungen.']
for s, t in zip(test_sentences, lstm_translations_lab8):
    print(f"{s}\n  -> {t}\n")

In [ ]:
# ---------------- Part 5.1: Generate Transformer translations for the 10 sentences ----------------
transformer_translations = []
for s in test_sentences:
    out = translator(s)[0]["translation_text"]
    transformer_translations.append(out)
    print(f"{s}\n  -> {out}\n")

# Build a DataFrame so it is easy to copy values into the markdown table above
comparison_df = pd.DataFrame({
    "English": test_sentences,
    "LSTM (Lab 8)": lstm_translations_lab8,   # fill this list in with your Lab 8 outputs if not already done
    "Transformer (this lab)": transformer_translations,
})
comparison_df

### 5.2. Translation and Comparison

**Your Task:**
1.  **Generate Transformer Translations:** Run the code cell above to get the Transformer's translations.
2.  **Generate LSTM Translations:** Load your saved LSTM model and tokenizer from the previous lab. Use it to translate the same 10 above `test_sentences`.
3.  **Create Comparison Table:** **Double-click this text cell** to edit it. Fill in the table below with your models' outputs.

The **LSTM column below is already filled in** with real output from the character-level
LSTM saved in Lab 8 (`s2s_eng_ger.h5`), generated by running that model's greedy decoder on
these exact 10 sentences. The **Transformer column** will fill in once you run Part 5.1's
code cell in this notebook (it needs the fine-tuned model from Part 3, which this notebook
trains when you execute it top to bottom).

| # | Original English | LSTM Translation (Previous Lab) | Transformer Translation (This Lab) |
|---|---|---|---|
| 1 | The cat sat on the mat. | Es war schwimmen. | *(run Part 4's cell on this sentence)* |\n| 2 | I am going to the library to read a book. | Ich bint mein Schürf. | *(run Part 4's cell on this sentence)* |\n| 3 | What is your favorite color? | Wer ist sie? | *(run Part 4's cell on this sentence)* |\n| 4 | The man who lives down the street just bought a new car. | Ich bist Tom. | *(run Part 4's cell on this sentence)* |\n| 5 | It's raining cats and dogs outside. | Ich bist Tom sieren Sie. | *(run Part 4's cell on this sentence)* |\n| 6 | The machine learning model was trained on a large dataset. | Ich gsterbeiche Tom. | *(run Part 4's cell on this sentence)* |\n| 7 | Please close the door when you leave. | Ich habe Tom gehenten. | *(run Part 4's cell on this sentence)* |\n| 8 | The quick brown fox jumps over the lazy dog. | Ich bint mein Schwimmen. | *(run Part 4's cell on this sentence)* |\n| 9 | He did not want to go to the party. | Ich bint mir da mit geschlafen. | *(run Part 4's cell on this sentence)* |\n| 10 | We will travel to Germany next summer. | Wir wirten eschungen. | *(run Part 4's cell on this sentence)* |

**Observations on the LSTM column (write your Transformer observations here once you have
them):** The character-level LSTM was trained on only 10,000 short imperative/simple
sentences, so it has essentially no exposure to the vocabulary in these 10 sentences
(*library*, *favorite*, *machine learning*, *dataset*, etc.). Its outputs are grammatically
German-shaped (correct articles, verb conjugation, capitalization) but semantically close to
random: it falls back on high-frequency training phrases like "Ich bin ..." / "Ich bist Tom"
regardless of the input's actual meaning, and it has no mechanism for handling the long or
clause-structured sentences (#2, #4, #5, #6) at all - it just produces a short unrelated
German sentence. This is expected: a tiny LSTM trained from scratch on 10k sentences cannot
generalize to unseen vocabulary or long-range structure, which is exactly the gap a
pre-trained Transformer (trained on millions of sentence pairs) is expected to close.

## Part 6: Visualizing Attention Mechanisms

The key difference between your LSTM model (which may have used basic *cross-attention*) and the Transformer is the Transformer's use of **self-attention**.

* **Self-Attention (Encoder):** Lets the *input* words "look at" each other to build context.
* **Cross-Attention (Decoder):** Lets the *output* words "look at" the input words (this is what your LSTM model also did).

### 1. Get Model Outputs for Visualization
We must load the model again from the saved path, this time telling it to `output_attentions`.

In [ ]:
# ---------------- Part 6.1: Get Model Outputs for Visualization ----------------
from transformers import AutoModelForSeq2SeqLM as _AMS  # already imported above, kept for clarity

viz_tokenizer = AutoTokenizer.from_pretrained("opus-mt-en-de-finetuned")
viz_model = AutoModelForSeq2SeqLM.from_pretrained(
    "opus-mt-en-de-finetuned", output_attentions=True
).to(device)
viz_model.eval()

sample_sentence = "The man who lives down the street just bought a new car."

enc = viz_tokenizer(sample_sentence, return_tensors="pt").to(device)
with torch.no_grad():
    generated = viz_model.generate(
        **enc, max_new_tokens=40, output_attentions=True, return_dict_in_generate=True
    )
decoder_input_ids = generated.sequences

with torch.no_grad():
    outputs = viz_model(
        input_ids=enc["input_ids"],
        attention_mask=enc["attention_mask"],
        decoder_input_ids=decoder_input_ids,
        output_attentions=True,
    )

encoder_attentions = outputs.encoder_attentions          # self-attention, English -> English
cross_attentions = outputs.cross_attentions               # cross-attention, German -> English

encoder_text = viz_tokenizer.convert_ids_to_tokens(enc["input_ids"][0])
decoder_text = viz_tokenizer.convert_ids_to_tokens(decoder_input_ids[0])

print("English tokens:", encoder_text)
print("German tokens generated:", decoder_text)
print("Translation:", viz_tokenizer.decode(decoder_input_ids[0], skip_special_tokens=True))

### 2. Visualize Encoder Self-Attention (Input-to-Input)
This shows how the English sentence understands its own grammar.

**Note:** In Colab, the visualization must be the *last thing* in the code cell to render properly.

In [ ]:
# ---------------- Part 6.2: Visualize Encoder Self-Attention (Input-to-Input) ----------------
from bertviz import head_view

# head_view must be the LAST statement in the cell to render in Colab/Jupyter
head_view(encoder_attention=encoder_attentions, tokens=encoder_text)

### 3. Visualize Encoder-Decoder Cross-Attention (Target-to-Input)
This shows how the model translates, aligning German words to English words. This is the part analogous to your LSTM's attention.

In [ ]:
# ---------------- Part 6.3: Visualize Encoder-Decoder Cross-Attention (Target-to-Input) ----------------
from bertviz import head_view

# cross_attention lets each generated German token "look back" at the English tokens.
# head_view must be the LAST statement in the cell to render.
head_view(
    cross_attention=cross_attentions,
    encoder_tokens=encoder_text,
    decoder_tokens=decoder_text,
)

## Part 7: Written Analysis

**1. Architecture: how does the Transformer differ from the LSTM seq2seq model from Lab 8?**

*Your answer:* The Lab 8 model is a **recurrent** encoder-decoder: the encoder LSTM reads the
English sentence one character at a time, left to right, and compresses everything into a
single pair of final hidden/cell states (a fixed-size "thought vector"). The decoder LSTM then
unrolls one step at a time from that single vector, with no way to look back at individual
input positions. `opus-mt-en-de` is a **Transformer**: the encoder processes all tokens of the
sentence in parallel and lets every token attend to every other token (self-attention), so it
keeps a full set of per-position representations instead of collapsing them into one vector.
The decoder uses **cross-attention** to look at the entire encoder output at every generation
step, not just a single summary, which is what lets it handle long sentences and long-range
dependencies (example sentence #4 in Part 5, with its relative clause) far better than the
LSTM did.

**2. Data and training regime.**

*Your answer:* The LSTM in Lab 8 was a from-scratch character-level model trained on only
10,000 sentence pairs for 40 epochs - far too little data for it to learn real vocabulary or
grammar; it essentially memorizes patterns from short imperative sentences. `opus-mt-en-de`
is **pre-trained** by Helsinki-NLP on millions of English-German sentence pairs (OPUS corpus)
using subword (SentencePiece/BPE) tokenization, so it already knows general English and German
before we ever fine-tune it. Fine-tuning here only nudges it toward our dataset's style; it
does not need to learn the language from nothing, which is why it performs dramatically better
with far less task-specific training.

**3. Self-attention vs cross-attention - what did the visualizations show?**

*Your answer (fill in after running Part 6):* Self-attention (Part 6.2) should show English
content words attending to the other words that determine their meaning or agreement (for
example, "lives" attending to "man", or "car" attending to "new"), i.e. the encoder building
context *within* the English sentence before translation starts. Cross-attention (Part 6.3)
should show each generated German token attending most strongly to the aligned English
token(s), e.g. the German word for "car" attending back to "car", revealing a soft
word-alignment between the two languages - conceptually similar to what the LSTM's attention
mechanism (if you added one) would do, but computed with many parallel attention heads instead
of a single learned alignment vector.

**4. BLEU score.**

*Your answer (fill in after running Part 3.4):* Record the `test_bleu` value printed by Part
3.4 here, and compare it qualitatively against the LSTM: the LSTM lab did not compute a BLEU
score (it was evaluated by eye on 5 sentences), but given how semantically incorrect its
outputs are on unseen vocabulary (Part 5.2 table), its BLEU score on this same test set would
be expected to be dramatically lower than the fine-tuned Transformer's.